In [ ]:
import os
import json
from dotenv import load_dotenv
from pydantic import (BaseModel, Field, ValidationError)
from typing import (List, Literal)
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)
from azure.storage.blob import BlobServiceClient
from langchain_openai import (ChatOpenAI, OpenAIEmbeddings)
from langgraph.graph import (StateGraph, START, END)
from langchain.agents import create_agent

In [ ]:
load_dotenv()

In [ ]:
azure_foundry_project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
azure_foundry_model_endpoint = os.getenv("AZURE_FOUNDRY_MODEL_ENDPOINT")
azure_foundry_model_name = os.getenv("AZURE_FOUNDRY_MODEL_NAME")
azure_foundry_embedding_model_name = os.getenv("AZURE_FOUNDRY_EMBEDDING_MODEL_NAME")

In [ ]:
credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [ ]:
llm = ChatOpenAI(
    model=azure_foundry_model_name,
    base_url=f"{azure_foundry_project_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
    use_responses_api=True,
)

In [ ]:
def load_employee_data():
    """Download employee JSON data from Azure Blob Storage."""
    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_EMPLOYEES"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_EMPLOYEES"]

    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"employees": [...]}.
        if isinstance(data, dict):
            data = data.get("employees")

        if not isinstance(data, list):
            raise ValueError(
                "Employee data must be a JSON list or an "
                "object containing an 'employees' list."
            )

        return data

    finally:
        credential.close()

In [ ]:
def load_task_data():
    """Download task JSON data from Azure Blob Storage."""

    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_TASKS"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_TASKS"]
    
    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"tasks": [...]}.
        if isinstance(data, dict):
            data = data.get("tasks")

        if not isinstance(data, list):
            raise ValueError(
                "Task data must be a JSON list or an "
                "object containing a 'tasks' list."
            )

        return data

    finally:
        credential.close()

In [ ]:
employee_data = load_employee_data()
employee_data

In [ ]:
task_data = load_task_data()
task_data

In [ ]:
def evaluate_employee_skill_set(employee_name, skill, level, evalation_agent):
    """
    This function returns an evaluation of a particular employee's skills.
    """

    evaluation_prompt = f"""
        The employee's name is {employee_name}.
        The skill under evaluation is {skill}.
        The level of expertise is {level}.
    """

    result = evaluation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    })
    return result

In [ ]:
evaluation_agent_system_prompt = (
    "You are an employee skill evaluator. "
    "Your job is to read an employee's skill as well as how advanced they are at that particular skill. "
    "You must output a brief statement about their level at that skill."
)

evaluation_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt
)

In [ ]:
class EmployeeEvaluation(BaseModel):
    employee_id: int
    evaluation: str

In [ ]:
evaluations: List[EmployeeEvaluation] = []
for employee in employee_data:
    for skill, level in employee['skill_map'].items():
        print('skill ', skill)
        print('level ', level)
        print('name ', employee['name'])
        evaluation = evaluate_employee_skill_set(employee['name'], skill, level, evaluation_agent)
        evaluation_content = evaluation['messages'][1].content
        evaluation_text = evaluation_content[0]['text']
        print('evaluation_text ', evaluation_text)
        evaluations.append({
            "employee_id": employee["id"],
            "evaluation": evaluation_text
        })

In [ ]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(
    model=azure_foundry_embedding_model_name,
    base_url=f"{azure_foundry_model_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

In [ ]:
evaluations

In [ ]:
class SkillRelationship(BaseModel):
    skill_1: str
    skill_2: str

    overlap_level: Literal["None", "Weak", "Moderate", "Strong"] = Field(
        description="The degree of technical or conceptual overlap between the two skills."
    )

    shared_concepts: list[str] = Field(
        description="Technical concepts, technologies, or knowledge shared by the two skills."
    )

    transferable_knowledge: str = Field(
        description="Explain what knowledge of one skill could transfer to the other."
    )

    limitations: str = Field(
        description="Explain what knowledge of one skill does not establish about proficiency in the other."
    )


class SkillSimilarityReport(BaseModel):
    relationships: list[SkillRelationship] = Field(
        description="Relationships identified between the supplied skills."
    )

In [ ]:
class TaskEmployeeCandidate(BaseModel):
    """
        This class is designed for evaluating a particular skill for an individual employee. 
        `Low` means limited opportunity; `High` means significant opportunity.
    """
    learning_opportunity: Literal["Low", "Medium", "High"] = Field(
        description="Employee's learning opportunity."
    )

    leadership_capability: Literal["Low", "Medium", "High"] = Field(
        description="Employee's technical leadership and mentoring capability."
    )

    learning_explanation: str = Field(
        description="Brief explanation supporting the learning opportunity classification."
    )

    leadership_explanation: str = Field(
        description="Brief explanation supporting the leadership capability classification."
    )

In [ ]:
skill_matching_agent_system_prompt = (
    "You are a technical skill evaluation and task-to-employee matching agent. "

    "YOUR OBJECTIVE: "
    "Evaluate an employee's suitability for a task by comparing the "
    "required technical skills against the employee's existing skills "
    "and proficiency levels. Identify learning opportunities and "
    "potential technical leadership capabilities. "

    "SKILL EVALUATION: "
    "Evaluate each required skill against the employee's skill map. "
    "Use demonstrated proficiency as your primary source of evidence. "
    "You will also receive findings from a skill similarity investigator. "
    "Use these findings to identify transferable knowledge between "
    "related technologies, frameworks, and concepts. "

    "For example, Angular experience may provide a foundation for "
    "understanding CSS because Angular applications commonly use CSS "
    "for styling. However, Angular proficiency does not automatically "
    "establish advanced CSS proficiency. "
    "Distinguish demonstrated proficiency from inferred proficiency. "
    "Never invent skills, experience, or qualifications. "

    "Consider the strength of the relationship between skills. "
    "Strong overlap may indicate meaningful transferable knowledge, "
    "while weak overlap should have limited influence on your evaluation. "
    "Do not treat related skills as interchangeable. "

    "LEARNING OPPORTUNITY: "
    "Classify the employee's opportunity to learn and develop the "
    "technical skills required by the task. "
    "LOW: The employee demonstrates advanced or expert proficiency "
    "in most required skills and has limited learning opportunities. "
    "MEDIUM: The employee demonstrates a mixture of proficiency levels "
    "or has some opportunities to improve existing skills. "
    "HIGH: The employee demonstrates beginner-level proficiency in "
    "multiple required skills and has substantial learning opportunities. "

    "LEADERSHIP CAPABILITY: "
    "Classify the employee's potential to provide technical leadership, "
    "mentorship, or guidance to others working on the task. "
    "LOW: The employee demonstrates beginner-level proficiency in most "
    "required skills and has limited demonstrated ability to guide others. "
    "MEDIUM: The employee demonstrates intermediate proficiency or "
    "advanced proficiency in some required skills and may provide "
    "technical guidance in those areas. "
    "HIGH: The employee demonstrates advanced or expert proficiency "
    "in most required skills and has strong potential to mentor others. "

    "DEFAULT CLASSIFICATION RULE: "
    "When in doubt, always default to MEDIUM for both classifications. "
    "If a required skill is missing from the employee's skill map, "
    "treat its proficiency as unknown rather than beginner-level. "
    "Missing information alone is not evidence of a significant learning "
    "opportunity or a lack of leadership capability. "
    "Only assign LOW or HIGH when the available evidence provides "
    "a reasonable basis for doing so. "

    "CLASSIFICATION CONSIDERATIONS: "
    "Evaluate learning opportunity and leadership capability independently. "
    "An employee may have both HIGH learning opportunity and HIGH "
    "leadership capability when they demonstrate expertise in some "
    "required skills but have opportunities to develop in others. "
    "Consider the overall alignment between task requirements, the "
    "employee's skill map, and the similarity investigator's findings. "

    "STRICT OUTPUT REQUIREMENTS: "
    "For learning_opportunity and leadership_capability, return exactly "
    "one of these values: 'Low', 'Medium', or 'High'. "
    "Use the exact capitalization shown. "
    "Never return numerical scores, percentages, or alternative labels. "

    "EXPLANATION REQUIREMENTS: "
    "Provide a brief, evidence-based explanation for each classification. "
    "Reference specific required skills, demonstrated proficiency, "
    "and relevant transferable knowledge where applicable. "
    "Clearly distinguish demonstrated skills from inferred knowledge. "
    "Do not make unsupported assumptions about the employee's abilities. "

    "Your final response must conform to the required structured output schema."
)

skill_matching_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=skill_matching_agent_system_prompt,
    response_format=TaskEmployeeCandidate
)

In [ ]:
skill_similarity_investigator_prompt = (
    "You are a technical skill similarity investigator. "

    "YOUR OBJECTIVE: "
    "Analyze a list of technical skills and identify relationships, "
    "similarities, shared concepts, and transferable knowledge between them. "

    "SKILL ANALYSIS: "
    "Examine the technical and conceptual relationships between the supplied "
    "skills. Identify whether knowledge of one skill could provide a useful "
    "foundation for understanding or learning another. "

    "Consider shared technologies, programming concepts, frameworks, "
    "languages, architectures, tools, and development practices. "

    "For example, Angular and CSS have a meaningful relationship because "
    "Angular applications commonly use CSS for styling. An individual with "
    "Angular experience may have transferable knowledge relevant to CSS. "

    "However, related skills are not necessarily interchangeable. "
    "Angular proficiency does not automatically establish advanced CSS "
    "proficiency. Clearly distinguish transferable knowledge from "
    "demonstrated proficiency. "

    "OVERLAP CLASSIFICATION: "
    "STRONG: The skills share substantial technical concepts or practical "
    "knowledge. Experience in one provides a strong foundation for the other. "
    "MODERATE: The skills share some concepts or practices, but also require "
    "meaningful independent knowledge. "
    "WEAK: The skills have limited conceptual or technical overlap. "
    "NONE: There is no meaningful relationship between the skills. "

    "ANALYSIS REQUIREMENTS: "
    "Evaluate relationships objectively. Do not invent similarities simply "
    "because two skills are used in the same industry or project. "
    "Do not assume that proficiency in one skill guarantees proficiency "
    "in another. "
    "Identify only meaningful relationships between the supplied skills. "

    "OUTPUT REQUIREMENTS: "
    "Return the identified relationships using the required structured "
    "output schema. "
    "Provide concise explanations of shared concepts, transferable knowledge, "
    "and the limitations of that knowledge. "
)

skill_similarity_investigator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=skill_similarity_investigator_prompt,
    response_format=SkillSimilarityReport
)

In [ ]:
def match_task_to_skills(
    task_id: int,
    skills: list[str],
    employee: dict,
    skill_similarity_investigator_agent,
    skill_matching_agent
) -> tuple[TaskEmployeeCandidate, int, int]:

    print("Task ID:", task_id)
    print("Required skills:", skills)
    print("Employee:", employee)

    # STEP 1: Investigate skill relationships
    try:
        required_skill_names = {
            skill.strip().casefold()
            for skill in skills
        }
    
        employee_skills_for_similarity = {
            skill: proficiency
            for skill, proficiency in employee["skill_map"].items()
            if skill.strip().casefold() not in required_skill_names
        }
        
        similarity_prompt = f"""
        Required task skills:
        {skills}
        
        Employee skills that are different from the required task skills:
        {employee_skills_for_similarity}
        
        Identify meaningful relationships between the required task skills
        and the employee's existing skills.
        
        IMPORTANT:
        - Compare each required task skill only against distinct employee skills.
        - Do not compare a skill against itself.
        - Skill names that differ only by capitalization or whitespace
          represent the same skill.
        - Do not report exact skill matches as transferable knowledge.
        - Focus on meaningful relationships between different technologies,
          frameworks, and concepts.
        - If no meaningful relationships exist, return an empty relationships list.
        """

        similarity_result = skill_similarity_investigator_agent.invoke({
            "messages": [
                {"role": "user", "content": similarity_prompt}
            ]
        })

        similarity_report = similarity_result["structured_response"]

        similarity_findings = similarity_report.model_dump_json(indent=2)

        print("Similarity findings:", similarity_findings)

    except Exception as e:
        print(f"Skill similarity investigation failed: {e}")

        # Continue matching without similarity findings.
        similarity_findings = (
            "No similarity findings are available. "
            "Evaluate the employee using the available skill map "
            "and demonstrated proficiency."
        )

    # STEP 2: Evaluate employee against task requirements
    skill_matching_prompt = f"""
    Required task skills:
    {skills}

    Employee skill map:
    {employee['skill_map']}

    Skill similarity investigation:
    {similarity_findings}

    Evaluate the employee's learning opportunities and
    leadership capabilities in relation to this task.

    Consider demonstrated proficiency and relevant transferable
    knowledge. Do not assume that related skills establish
    proficiency without supporting evidence.
    """

    try:
        result = skill_matching_agent.invoke({
            "messages": [
                {"role": "user", "content": skill_matching_prompt}
            ]
        })

        structured_response = result["structured_response"]

        print("Structured response:", structured_response)

        return (
            structured_response,
            task_id,
            employee["id"]
        )

    except Exception as e:
        print(f"Skill matching agent failed: {e}")

        return (
            TaskEmployeeCandidate(
                learning_opportunity="Medium",
                leadership_capability="Medium",
                learning_explanation="Unable to determine learning opportunity.",
                leadership_explanation="Unable to determine leadership capability."
            ),
            task_id,
            employee["id"]
        )

In [ ]:
employee_data

In [ ]:
skill_matches = []
for task in task_data:
    skills = ", ".join(task['required_skills'])
    for employee in employee_data:
        skill_match = match_task_to_skills(task['id'], skills, employee, skill_similarity_investigator_agent, skill_matching_agent)
        skill_matches.append(skill_match)

In [ ]:
for skill_match in skill_matches:
    print(skill_match)
    print('-')

In [ ]:
SCORE_MAP = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

for skill_match in skill_matches:
    learning_score = SCORE_MAP[skill_match[0].learning_opportunity]
    leadership_score = SCORE_MAP[skill_match[0].leadership_capability]
    print('learning_score ', learning_score)
    print('leadership_score ', leadership_score)

In [ ]:
learning_ranked = sorted(
    skill_matches,
    key=lambda x: SCORE_MAP[x[0].learning_opportunity]
)
learning_ranked

In [ ]:
leadership_ranked = sorted(
    skill_matches,
    key=lambda x: SCORE_MAP[x[0].leadership_capability],
)
leadership_ranked

In [ ]:
print(len(leadership_ranked))
print(len(learning_ranked))